# Overbooking: how many reservations to sell?

An airline with $150$ seats must choose a fare and how many reservations to accept. Some booked passengers fail to show — here the no-show *fraction* is random, drawn from a Beta distribution with mean $0.10$ and standard deviation $0.025$ — so selling exactly $150$ reservations tends to fly with empty seats. Selling *more* than $150$ risks having to bump passengers, which is costly.

The economics of a reservation are asymmetric:

- A **no-show** is refunded a fraction (`REFUND_FACTOR` $= 0.5$) of the fare.
- A **bumped** passenger (when show-ups exceed capacity) must be compensated at a multiple of the fare (`COMPENSATION_FACTOR` $= 2.5$).

We jointly optimize price and number of reservations by Monte Carlo over random demand and no-show draws, then compare against the policy of never overbooking (reservations capped at capacity). The gap measures what overbooking is worth.

## Setup and parameters

Fares, capacity, the compensation/refund factors, the linear demand curve, and the no-show distribution. We then convert the no-show mean and standard deviation into Beta shape parameters $(a, b)$ and draw the random demand-forecast errors and no-show fractions once, so every candidate policy is evaluated on the same realizations.

In [1]:
import numpy as np
from scipy.stats import beta, norm
from scipy.optimize import minimize
np.random.seed(666)

In [2]:
# Constants
COST = 12000
CAPACITY = 150
COMPENSATION_FACTOR = 2.5
REFUND_FACTOR = 0.5
DEMAND_INTERCEPT = 600
DEMAND_COEFFICIENT = -5
DEMAND_STD = 10
NOSHOW_MEAN = 0.1
NOSHOW_STD = 0.025
NUM_REALIZATIONS = 1000

In [3]:
# Convert mean and standard deviation to a and b parameters for the beta distribution
NOSHOW_A = ((1 - NOSHOW_MEAN) / NOSHOW_STD**2 - 1 / NOSHOW_MEAN) * NOSHOW_MEAN**2
NOSHOW_B = NOSHOW_A * (1 / NOSHOW_MEAN - 1)

# Random realizations
realization_forecast_error = np.random.normal(0, DEMAND_STD, NUM_REALIZATIONS)
realization_no_shows_fraction = beta.rvs(NOSHOW_A, NOSHOW_B, size=NUM_REALIZATIONS)

## Jointly optimize price and reservations

`mean_profit(price, reservations)` averages profit over the random draws: revenue from sales, minus the fixed cost, minus refunds to no-shows, minus compensation whenever show-ups exceed capacity. We maximize it over both the fare and the number of reservations.

In [4]:
def mean_profit(price, reservations):
    profits = []
    for i in range(NUM_REALIZATIONS):
        demand = DEMAND_INTERCEPT + DEMAND_COEFFICIENT * price + realization_forecast_error[i]
        sales = np.minimum(reservations, demand)
        no_shows = sales * realization_no_shows_fraction[i]
        passengers = sales - no_shows
        # Bumped passengers are compensated only when show-ups exceed capacity
        if passengers > CAPACITY:
            compensation = (passengers - CAPACITY) * price * COMPENSATION_FACTOR
        else:
            compensation = 0
        refund = no_shows * price * REFUND_FACTOR
        revenue = sales * price
        cost = COST + compensation + refund
        profits.append(revenue - cost)
    return np.mean(profits)

def objective(x):
    return -mean_profit(x[0], x[1])  # negate: maximize profit via a minimizer

x0 = [100, 150]
bounds = [(0, None), (0, None)]  # price and reservations are non-negative
res = minimize(objective, x0, bounds=bounds)

In [5]:
print(f"Optimal price: ${round(res.x[0])}")
print(f"Optimal number of reservations: {round(res.x[1])}")

Optimal price: $87
Optimal number of reservations: 165


## Benchmark: never overbook

For comparison, fix reservations at capacity ($150$) and optimize the fare alone. This is the best the airline can do if it refuses to overbook. The difference in profit from the joint solution above is the value of overbooking.

In [6]:
def profit_fixed_reservations(price):
    return mean_profit(price, CAPACITY)  # reservations fixed at capacity: no overbooking

def objective_fixed_reservations(x):
    return -profit_fixed_reservations(x)

res_fixed_reservations = minimize(objective_fixed_reservations, x0[0])

In [7]:
print(f"Optimal price with fixed reservations: ${round(res_fixed_reservations.x[0])}")

Optimal price with fixed reservations: $89


## Sensitivity: profit at a few nearby policies

A quick look at how profit responds to small changes around the optimum. Overbooking to $166$ reservations at the optimal fare earns far more than holding reservations at capacity, confirming that the gain comes from selling past $150$ rather than from the exact fare.

In [8]:
round(mean_profit(87, 166))   # optimal fare, overbook to 166

1175

In [9]:
round(mean_profit(87, 150))   # optimal fare, no overbooking

375

In [10]:
round(mean_profit(89, 150))   # fixed-reservations fare, no overbooking

516